In [0]:
dbutils.widgets.text("catalog", "global_airline", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
dbutils.widgets.text("pipeline_run_id", "ADHOC", "Pipeline run ID")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

In [0]:
# B1: Load airline.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.airline') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.airline')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/airline/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B1: Load passenger.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.passenger') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.passenger')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/passenger/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B1: Load route.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.route') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.route')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/route/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B1: Load route.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.route') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.route')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/route/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B1: Load airline_route.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.airline_route') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.airline_route')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/airline_route/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B1: Load nation.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.nation') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.nation')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/nation/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B1: Load region.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.region') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _pipeline_run_id STRING
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.region')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at,
           '{pipeline_run_id}' AS _pipeline_run_id
    FROM '/Volumes/global_airline/bronze/raw/region/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
from pyspark.sql.functions import *

# B4: Load booking.csv

df_orders = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.schemaLocation", "/Volumes/global_airline/bronze/pipeline_state/booking_schema/")
    .load("/Volumes/global_airline/bronze/raw/booking/")
    .select(
        col("*"),
        col("_metadata.file_path").alias("_source_file"),
        current_timestamp().alias("_ingested_at"),
        lit(f"{pipeline_run_id}").alias("_pipeline_run_id")
    )
)

(
    df_orders.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/global_airline/bronze/pipeline_state/booking_checkpoint/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(f"{catalog}.bronze.booking")
)

In [0]:
from pyspark.sql.functions import *

# B4: Load flight_booking.csv

df_orders = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.schemaLocation", "/Volumes/global_airline/bronze/pipeline_state/flight_booking_schema/")
    .load("/Volumes/global_airline/bronze/raw/flight_booking/")
    .select(
        col("*"),
        col("_metadata.file_path").alias("_source_file"),
        current_timestamp().alias("_ingested_at"),
        lit(f"{pipeline_run_id}").alias("_pipeline_run_id")
    )
)

(
    df_orders.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/global_airline/bronze/pipeline_state/flight_booking_checkpoint/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(f"{catalog}.bronze.flight_booking")
)